# Train Neuron Classifier

This notebook loads manual annotations, extracts HKS features from neuron meshes, and trains a Random Forest classifier.

In [ ]:
import os
import pandas as pd
import numpy as np
import caveclient
from meshparty import trimesh_io
from meshmash import condensed_hks_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm.notebook import tqdm

# Constants
DATASTACK = 'minnie65_phase3_v1'
ANNOTATION_FILE = 'annotations.csv'
CACHE_DIR = 'meshes'

# Initialize Client
client = caveclient.CAVEclient(DATASTACK)
cv = client.info.segmentation_cloudvolume()
cv.progress = False

mm = trimesh_io.MeshMeta(cv=cv, disk_cache_path=CACHE_DIR)

## 1. Load Annotations

In [ ]:
if not os.path.exists(ANNOTATION_FILE):
    raise FileNotFoundError(f"{ANNOTATION_FILE} not found. Please run annotation_tool.ipynb first.")

df_labels = pd.read_csv(ANNOTATION_FILE)
print(f"Loaded {len(df_labels)} annotations.")
df_labels.head()

## 2. Feature Extraction (HKS)

## 3. Train Classifier

In [ ]:
# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples.")

# Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict
y_pred = clf.predict(X_test)

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

### Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importances = clf.feature_importances_
feature_names = X.columns
forest_importances = pd.Series(importances, index=feature_names)

fig, ax = plt.subplots(figsize=(10, 6))
forest_importances.sort_values(ascending=False).head(20).plot.bar(ax=ax)
ax.set_title("Top 20 Feature Importances")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()
plt.show()